# Inventory Domain Walkthrough

This notebook stays inside the inventory domain. It uses the public
`inventory/service.py` facade plus the internal formatters/helpers that turn
typed results into terminal-style output.


## Setup

The setup cell seeds a fresh database workspace and imports the local
package directly from `src/`.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'mtg_source_stack').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

WORK_DIR = REPO_ROOT / 'var' / 'notebooks' / '03_inventory_domain'
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards
from mtg_source_stack.inventory.report_formatters import (
    format_add_card_result,
    format_merge_rows_result,
    format_owned_rows,
    format_set_acquisition_result,
    format_set_location_result,
    format_set_notes_result,
    format_set_quantity_result,
    format_set_tags_result,
    format_split_row_result,
)
from mtg_source_stack.inventory.report_helpers import render_table
from mtg_source_stack.inventory.response_models import serialize_response
from mtg_source_stack.inventory.service import (
    add_card,
    create_inventory,
    list_owned,
    merge_rows,
    search_cards,
    set_acquisition,
    set_location,
    set_notes,
    set_quantity,
    set_tags,
    split_row,
)

DB_PATH = WORK_DIR / 'inventory_walkthrough.sqlite3'
SCRYFALL_JSON = WORK_DIR / 'scryfall_sample.json'
IDENTIFIERS_JSON = WORK_DIR / 'identifiers_sample.json'
PRICES_JSON = WORK_DIR / 'prices_sample.json'


def write_sample_payloads() -> None:
    scryfall_payload = [
        {
            'id': 's1',
            'oracle_id': 'o1',
            'name': 'Lightning Bolt',
            'set': 'lea',
            'set_name': 'Limited Edition Alpha',
            'collector_number': '161',
            'lang': 'en',
            'rarity': 'common',
            'released_at': '1993-08-05',
            'mana_cost': '{R}',
            'type_line': 'Instant',
            'oracle_text': 'Lightning Bolt deals 3 damage to any target.',
            'colors': ['R'],
            'color_identity': ['R'],
            'finishes': ['nonfoil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/lightning-bolt'},
            'tcgplayer_id': 534658,
        },
        {
            'id': 's2',
            'oracle_id': 'o2',
            'name': 'Sol Ring',
            'set': 'clb',
            'set_name': "Commander Legends: Battle for Baldur's Gate",
            'collector_number': '860',
            'lang': 'en',
            'rarity': 'rare',
            'released_at': '2022-06-10',
            'mana_cost': '{1}',
            'type_line': 'Artifact',
            'oracle_text': '{T}: Add {C}{C}.',
            'colors': [],
            'color_identity': [],
            'finishes': ['nonfoil', 'foil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/sol-ring'},
            'tcgplayer_id': 271111,
        },
        {
            'id': 's3',
            'oracle_id': 'o3',
            'name': 'Counterspell',
            'set': '7ed',
            'set_name': 'Seventh Edition',
            'collector_number': '67',
            'lang': 'en',
            'rarity': 'uncommon',
            'released_at': '2001-04-11',
            'mana_cost': '{U}{U}',
            'type_line': 'Instant',
            'oracle_text': 'Counter target spell.',
            'colors': ['U'],
            'color_identity': ['U'],
            'finishes': ['nonfoil'],
            'legalities': {'commander': 'legal'},
            'purchase_uris': {'tcgplayer': 'https://example.test/tcg/counterspell'},
            'tcgplayer_id': 333333,
        },
    ]

    identifiers_payload = {
        'data': {
            'uuid-1': {'identifiers': {'scryfallId': 's1', 'tcgplayerProductId': '534658'}},
            'uuid-2': {'identifiers': {'scryfallId': 's2', 'tcgplayerProductId': '271111'}},
            'uuid-3': {'identifiers': {'scryfallId': 's3', 'tcgplayerProductId': '333333'}},
        }
    }

    prices_payload = {
        'data': {
            'uuid-1': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 2.92}}}}},
            'uuid-2': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 1.75}, 'foil': {'2026-03-27': 4.25}}}}},
            'uuid-3': {'paper': {'tcgplayer': {'currency': 'USD', 'retail': {'normal': {'2026-03-27': 1.25}}}}},
        }
    }

    SCRYFALL_JSON.write_text(json.dumps(scryfall_payload), encoding='utf-8')
    IDENTIFIERS_JSON.write_text(json.dumps(identifiers_payload), encoding='utf-8')
    PRICES_JSON.write_text(json.dumps(prices_payload), encoding='utf-8')


def seed_catalog() -> None:
    initialize_database(DB_PATH)
    write_sample_payloads()
    import_scryfall_cards(DB_PATH, SCRYFALL_JSON)
    import_mtgjson_identifiers(DB_PATH, IDENTIFIERS_JSON)
    import_mtgjson_prices(DB_PATH, PRICES_JSON)


print('repo root:', REPO_ROOT)
print('work dir:', WORK_DIR)
print('db path:', DB_PATH)


## Create the Catalog and Inventory Container

This starts with a fresh catalog import, then creates one named inventory.
After that, `search_cards()` can resolve printings directly from the local
SQLite database.


In [ ]:
seed_catalog()
create_inventory(
    DB_PATH,
    slug='personal',
    display_name='Personal Collection',
    description='Binders, decks, and loose cards',
)

search_rows = serialize_response(
    search_cards(
        DB_PATH,
        query='Lightning Bolt',
        set_code='lea',
        rarity='common',
        finish='normal',
        lang='en',
        exact=False,
        limit=10,
    )
)

print(render_table(search_rows, [
    ('name', 'name'),
    ('set_code', 'set'),
    ('collector_number', 'number'),
    ('lang', 'lang'),
    ('rarity', 'rarity'),
    ('finishes', 'finishes'),
    ('scryfall_id', 'scryfall_id'),
]))


## Add Rows and Make Routine Edits

The service facade returns typed mutation models. The human-readable output
below comes from the same formatter helpers the CLI uses.


In [ ]:
bolt = add_card(
    DB_PATH,
    inventory_slug='personal',
    inventory_display_name=None,
    scryfall_id='s1',
    tcgplayer_product_id=None,
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=4,
    condition_code='NM',
    finish='normal',
    language_code='en',
    location='Red Binder',
    acquisition_price='2.00',
    acquisition_currency='USD',
    notes='Playset',
    tags='burn,trade',
)
ring = add_card(
    DB_PATH,
    inventory_slug='personal',
    inventory_display_name=None,
    scryfall_id='s2',
    tcgplayer_product_id=None,
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=1,
    condition_code='LP',
    finish='foil',
    language_code='en',
    location='Commander Deck Box',
    acquisition_price='3.50',
    acquisition_currency='USD',
    notes='One shiny copy',
    tags='commander',
)
counterspell = add_card(
    DB_PATH,
    inventory_slug='personal',
    inventory_display_name=None,
    scryfall_id='s3',
    tcgplayer_product_id=None,
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=2,
    condition_code='NM',
    finish='normal',
    language_code='en',
    location='Blue Box',
    acquisition_price=None,
    acquisition_currency=None,
    notes=None,
    tags=None,
)

bolt_quantity = set_quantity(DB_PATH, inventory_slug='personal', item_id=bolt.item_id, quantity=3)
bolt_tags = set_tags(DB_PATH, inventory_slug='personal', item_id=bolt.item_id, tags='burn,staple')
bolt_notes = set_notes(DB_PATH, inventory_slug='personal', item_id=bolt.item_id, notes='Front binder copy')
bolt_location = set_location(DB_PATH, inventory_slug='personal', item_id=bolt.item_id, location='Red Binder / Front')
bolt_acquisition = set_acquisition(
    DB_PATH,
    inventory_slug='personal',
    item_id=bolt.item_id,
    acquisition_price='2.25',
    acquisition_currency='USD',
)

print(format_add_card_result(serialize_response(bolt)))
print()
print(format_add_card_result(serialize_response(ring)))
print()
print(format_set_quantity_result(serialize_response(bolt_quantity)))
print()
print(format_set_tags_result(serialize_response(bolt_tags)))
print()
print(format_set_notes_result(serialize_response(bolt_notes)))
print()
print(format_set_location_result(serialize_response(bolt_location)))
print()
print(format_set_acquisition_result(serialize_response(bolt_acquisition)))


## Split and Merge Rows

`split_row()` creates a second identity or merges into an existing one. The
`merge_rows()` command then combines two known rows explicitly.


In [ ]:
split_result = split_row(
    DB_PATH,
    inventory_slug='personal',
    item_id=counterspell.item_id,
    quantity=1,
    condition_code=None,
    finish=None,
    language_code=None,
    location='Deck Box',
)
merge_result = merge_rows(
    DB_PATH,
    inventory_slug='personal',
    source_item_id=split_result.item_id,
    target_item_id=counterspell.item_id,
)

print(format_split_row_result(serialize_response(split_result)))
print()
print(format_merge_rows_result(serialize_response(merge_result)))


## Inspect the Stored Rows and Audit Trail

The inventory view is still stored as current positions, but every mutation
now leaves behind an audit record with before/after snapshots.


In [ ]:
owned_rows = serialize_response(list_owned(DB_PATH, inventory_slug='personal', provider='tcgplayer', limit=None))
with connect(DB_PATH) as connection:
    audit_rows = []
    for row in connection.execute(
        """
        SELECT action, item_id, actor_type, request_id, metadata_json
        FROM inventory_audit_log
        ORDER BY id
        """
    ):
        metadata = json.loads(row['metadata_json'])
        audit_rows.append(
            {
                'action': row['action'],
                'item_id': row['item_id'],
                'actor_type': row['actor_type'],
                'request_id': row['request_id'],
                'role': metadata.get('role', ''),
            }
        )

print('Owned rows')
print(format_owned_rows(owned_rows))
print()
print('Audit trail')
print(render_table(audit_rows, [('action', 'action'), ('item_id', 'item_id'), ('actor_type', 'actor_type'), ('request_id', 'request_id'), ('role', 'role')]))
